In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn scipy pandas sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requi

In [2]:
%%writefile single_task_albert_qqp.py
import os
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import pearsonr, spearmanr


MODEL_NAME = "albert-base-v2"
TASK_NAME = "qqp"
SETTING = "single_task"

BATCH_SIZE = 32
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_LENGTH = 256
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 3
KEEP_LAST_K_CHECKPOINTS = 2
SEED = 42

BASE_DIR = Path("/content/drive/MyDrive/single_task_runs")
RUN_NAME = f"{SETTING}_{MODEL_NAME.replace('/', '_')}_{TASK_NAME}"
RUN_DIR = BASE_DIR / RUN_NAME

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
BEST_MODEL_DIR = RUN_DIR / "best_model"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
LOG_CSV_PATH = RUN_DIR / "history.csv"
LOG_JSON_PATH = RUN_DIR / "history.json"
STATE_PATH = RUN_DIR / "training_state.json"


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params


def get_task_info(task_name):
    if task_name == "sst2":
        return {
            "task_type": "classification",
            "dataset_loader": ("glue", "sst2"),
            "text_cols": ("sentence", None),
            "label_col": "label",
            "num_labels": 2,
            "primary_metric": "accuracy",
        }
    elif task_name == "qqp":
        return {
            "task_type": "classification",
            "dataset_loader": ("glue", "qqp"),
            "text_cols": ("question1", "question2"),
            "label_col": "label",
            "num_labels": 2,
            "primary_metric": "accuracy",
        }
    elif task_name == "stsb":
        return {
            "task_type": "regression",
            "dataset_loader": ("glue", "stsb"),
            "text_cols": ("sentence1", "sentence2"),
            "label_col": "label",
            "num_labels": 1,
            "primary_metric": "pearson",
        }
    else:
        raise ValueError("Unsupported task. Use sst2, qqp, or stsb.")


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_history(history):
    pd.DataFrame(history).to_csv(LOG_CSV_PATH, index=False)
    save_json(history, LOG_JSON_PATH)


def get_latest_checkpoint():
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")] if CHECKPOINT_DIR.exists() else []
    if not ckpts:
        return None
    return sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))[-1]


def cleanup_old_checkpoints(keep_k=2):
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")]
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))
    while len(ckpts) > keep_k:
        shutil.rmtree(ckpts.pop(0), ignore_errors=True)


def save_checkpoint(epoch, model, tokenizer, optimizer, scheduler, history, best_metric_so_far, patience_counter):
    ckpt_dir = CHECKPOINT_DIR / f"epoch_{epoch}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    torch.save(
        {
            "epoch": epoch,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history": history,
        },
        ckpt_dir / "trainer_state.pt",
    )
    save_json(
        {
            "epoch": epoch,
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history_length": len(history),
        },
        STATE_PATH,
    )
    cleanup_old_checkpoints(KEEP_LAST_K_CHECKPOINTS)


def get_na():
    return None


def load_and_prepare_data(tokenizer, task_info):
    dataset_name, subset_name = task_info["dataset_loader"]
    raw = load_dataset(dataset_name, subset_name)

    text_a, text_b = task_info["text_cols"]
    label_col = task_info["label_col"]
    task_type = task_info["task_type"]

    def preprocess_fn(examples):
        if text_b is None:
            enc = tokenizer(examples[text_a], truncation=True, max_length=MAX_LENGTH)
        else:
            enc = tokenizer(examples[text_a], examples[text_b], truncation=True, max_length=MAX_LENGTH)

        labels = examples[label_col]
        enc["labels"] = labels if task_type == "classification" else [float(x) for x in labels]
        return enc

    encoded = raw.map(preprocess_fn, batched=True, remove_columns=raw["train"].column_names)
    val_split = "validation" if "validation" in encoded else "validation_matched"
    return encoded["train"], encoded[val_split]


@torch.no_grad()
def evaluate_classification(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(batch["labels"].cpu().numpy().tolist())

    return {
        "eval_loss": total_loss / max(len(dataloader), 1),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "pearson": get_na(),
        "spearman": get_na(),
    }


@torch.no_grad()
def evaluate_regression(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()
        preds = outputs.logits.squeeze(-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(batch["labels"].cpu().numpy().tolist())

    pearson = float(pearsonr(all_labels, all_preds)[0]) if len(set(all_labels)) > 1 and len(set(all_preds)) > 1 else 0.0
    spearman = float(spearmanr(all_labels, all_preds)[0]) if len(set(all_labels)) > 1 and len(set(all_preds)) > 1 else 0.0

    return {
        "eval_loss": total_loss / max(len(dataloader), 1),
        "accuracy": get_na(),
        "macro_f1": get_na(),
        "precision": get_na(),
        "recall": get_na(),
        "pearson": pearson,
        "spearman": spearman,
    }


def main():
    set_seed(SEED)
    ensure_dirs()
    task_info = get_task_info(TASK_NAME)
    task_type = task_info["task_type"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"MODEL={MODEL_NAME} | TASK={TASK_NAME} | TYPE={task_type} | DEVICE={device}")
    print("Training continues until early stopping patience=3 is reached.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    train_ds, val_ds = load_and_prepare_data(tokenizer, task_info)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=task_info["num_labels"],
        problem_type="regression" if task_type == "regression" else None,
    ).to(device)

    trainable_params, total_params = count_parameters(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = len(train_loader)
    warmup_steps = max(1, int(steps_per_epoch * WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=10**12)

    history = []
    start_epoch = 1
    best_metric_so_far = -float("inf")
    patience_counter = 0

    latest_ckpt = get_latest_checkpoint()
    if latest_ckpt is not None:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        model = AutoModelForSequenceClassification.from_pretrained(latest_ckpt).to(device)
        state = torch.load(latest_ckpt / "trainer_state.pt", map_location=device)
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        history = state.get("history", [])
        best_metric_so_far = state.get("best_metric_so_far", -float("inf"))
        patience_counter = state.get("patience_counter", 0)
        start_epoch = state.get("epoch", 0) + 1

    epoch = start_epoch
    while True:
        model.train()
        start_time = time.time()
        running_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

            if step % 500 == 0 or step == len(train_loader):
                print(f"Epoch {epoch} | Step {step}/{len(train_loader)} | Train Loss: {running_loss/step:.6f}")

        train_loss = running_loss / max(len(train_loader), 1)
        eval_metrics = evaluate_classification(model, val_loader, device) if task_type == "classification" else evaluate_regression(model, val_loader, device)

        current_metric = eval_metrics["accuracy"] if task_type == "classification" else eval_metrics["pearson"]
        is_new_best = current_metric > best_metric_so_far

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0
            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)
            model.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
        else:
            patience_counter += 1

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "eval_loss": float(eval_metrics["eval_loss"]),
            "accuracy": eval_metrics["accuracy"],
            "macro_f1": eval_metrics["macro_f1"],
            "precision": eval_metrics["precision"],
            "recall": eval_metrics["recall"],
            "pearson": eval_metrics["pearson"],
            "spearman": eval_metrics["spearman"],
            "time_per_epoch": float(time.time() - start_time),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": MODEL_NAME,
            "dataset_name": TASK_NAME,
            "setting": SETTING,
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        history.append(row)
        save_history(history)
        save_checkpoint(epoch, model, tokenizer, optimizer, scheduler, history, best_metric_so_far, patience_counter)

        print(json.dumps(row, indent=2, ensure_ascii=False, default=str))

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

        epoch += 1

    if FINAL_MODEL_DIR.exists():
        shutil.rmtree(FINAL_MODEL_DIR)
    model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    save_history(history)

    print(f"History CSV : {LOG_CSV_PATH}")
    print(f"History JSON: {LOG_JSON_PATH}")
    print(f"Best model  : {BEST_MODEL_DIR}")
    print(f"Final model : {FINAL_MODEL_DIR}")
    print(f"Checkpoints : {CHECKPOINT_DIR}")


if __name__ == "__main__":
    main()

Writing single_task_albert_qqp.py


In [3]:
!python /content/single_task_albert_qqp.py

MODEL=albert-base-v2 | TASK=qqp | TYPE=classification | DEVICE=cuda
Training continues until early stopping patience=3 is reached.
config.json: 100% 684/684 [00:00<00:00, 1.28MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 140kB/s]
spiece.model: 100% 760k/760k [00:00<00:00, 12.4MB/s]
tokenizer.json: 1.31MB [00:00, 10.6MB/s]
README.md: 35.3kB [00:00, 57.5MB/s]
qqp/train-00000-of-00001.parquet: 100% 33.6M/33.6M [00:00<00:00, 36.8MB/s]
qqp/validation-00000-of-00001.parquet: 100% 3.73M/3.73M [00:00<00:00, 9.10MB/s]
qqp/test-00000-of-00001.parquet: 100% 36.7M/36.7M [00:00<00:00, 60.2MB/s]
Generating train split: 100% 363846/363846 [00:00<00:00, 840551.39 examples/s]
Generating validation split: 100% 40430/40430 [00:00<00:00, 962885.61 examples/s]
Generating test split: 100% 390965/390965 [00:00<00:00, 956326.84 examples/s]
Map: 100% 363846/363846 [00:57<00:00, 6331.39 examples/s]
Map: 100% 40430/40430 [00:13<00:00, 3079.16 examples/s]
Map: 100% 390965/390965 [01:02<00:00, 6301.42 